### **🔗 Lineage Graph (Data Flow DAG) in dbt**


-----------

In dbt, the **Lineage Graph** (also called **Data Flow DAG**) is a **visual map** that shows **how data flows** from raw sources → staging models → marts → downstream assets.

> **DAG = Directed Acyclic Graph**

> - **Directed** → data flows in one direction

> - **Acyclic** → no loops

> - **Graph** → nodes + edges

-----------

#### **1️⃣ What exactly is a Lineage Graph?**

A lineage graph answers **three critical questions** instantly:

- **Where does this data come from?** (upstream)

- **What depends on this model?** (downstream)

- **What breaks if this model changes?** (impact analysis)

In dbt:

- **Nodes** = sources, models, snapshots, seeds

- **Edges (arrows)** = dependencies created using `ref()` and `source()`

-------------

#### **2️⃣ How dbt builds the lineage (important concept)**

dbt **does NOT parse raw SQL logic.**

It builds lineage **only from dbt functions:**

🔹 `source()` → **raw data entry point**

In [ ]:
select *
from {{ source('airbnb', 'hosts') }}

🔹 `ref()` → **dependency between models**

In [ ]:
select *
from {{ ref('stg_hosts') }}

📌 If you don’t use `ref()` or `source()`, **dbt cannot track lineage.**

------------

#### **3️⃣ Simple end-to-end example (mentally visualize)**

**Raw Source**

In [ ]:
sources:
  - name: airbnb
    tables:
      - name: hosts

**Staging model**

In [ ]:
-- stg_hosts.sql

select
  id as host_id,
  name,
  created_at
from {{ source('airbnb', 'hosts') }}

**Dimension model**

In [ ]:
-- dim_hosts.sql

select
  host_id,
  name
from {{ ref('stg_hosts') }}

**Fact model**

In [ ]:
-- fct_listings.sql

select
  host_id,
  count(*) as total_listings
from {{ ref('dim_hosts') }}
group by host_id

**📊 Lineage graph looks like:**

In [ ]:
airbnb.hosts
      ↓
  stg_hosts
      ↓
  dim_hosts
      ↓
 fct_listings

---------

#### **4️⃣ Where you see the lineage graph**

You can view it in:

In [ ]:
dbt docs generate

dbt docs serve

Inside the **dbt docs UI**, click:

- a model → **Lineage tab**

- zoom in/out

- expand upstream & downstream

This UI is provided by **dbt Labs.**

-----------

#### **5️⃣ Upstream vs Downstream (must-know)**

**🔼 Upstream**

Everything **this model depends on**

- Sources

- Staging models

- Intermediate models

👉 Used for **debugging wrong data**


**🔽 Downstream**

Everything **that depends on this model**

- Dimensions

- Facts

- Final marts

- Dashboards (via exposures)

👉 Used for **impact analysis**

--------------

#### **6️⃣ Why lineage is extremely powerful (real-world use)**

**✅ Impact analysis**

> “If I change this column, what breaks?”

Just click the model → see **all downstream nodes.**

------

**✅ Faster debugging**

If a final dashboard is wrong:

- Open the fact model

- Trace upstream

- Find where bad data entered

---------

**✅ Safer deployments**

Before merging a PR:

- Check lineage

- Understand blast radius

------------

**7️⃣ Lineage + tests + docs (together)**

Lineage becomes **much more valuable** when combined with:

- **Tests** → trust indicators on nodes

- **Docs** → business meaning of each node

Example:

- Node shows ❌ failing test

- You instantly see:

    - where data came from

    - where bad data will flow

------------

#### **8️⃣ What lineage does NOT show (important clarity)**

Lineage graph does **not** show:

- Column-level transformations

- SQL logic inside models

- Data volume or performance

It shows **model-level dependency only.**

-------------

#### **9️⃣ Best practices for clean lineage ⭐**

**✔ Always use `ref()` and `source()`**

❌ Hardcoding schema.table names breaks lineage

✔ `ref()` keeps lineage + environment safety

--------

**✔ Layered modeling**

In [ ]:
sources → staging → intermediate → marts

This creates a **clean, readable DAG.**

-------

**✔ One responsibility per model**

Smaller models → clearer lineage → easier debugging.